## 2.1 理论计算题

输入一张大小为 3×32×32（通道数 × 高 × 宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 3×5×5。设定填充（Padding）为 2，步幅（Stride）为 2。

### 问题1：计算该卷积层输出的特征图（Feature Map）的尺寸（通道数 × 高 × 宽）

In [44]:
# 已知参数
input_channels = 3
input_height = 32
input_width = 32

num_kernels = 16
kernel_channels = 3
kernel_height = 5
kernel_width = 5

padding = 2
stride = 2

# 输出通道数等于卷积核数量
output_channels = num_kernels

# 输出高度计算公式: H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
output_height = (input_height + 2 * padding - kernel_height) // stride + 1

# 输出宽度计算公式: W_out = floor((W_in + 2*padding - kernel_size) / stride) + 1
output_width = (input_width + 2 * padding - kernel_width) // stride + 1

print(f"输出特征图尺寸: {output_channels} x {output_height} x {output_width}")

输出特征图尺寸: 16 x 16 x 16


### 问题2：计算这个卷积操作中，单个输出通道的一个像素值，需要对输入进行多少次点乘（乘法）操作？

In [45]:
# 单个卷积核的参数数量 = 输入通道数 x 卷积核高度 x 卷积核宽度
# 每个输出像素需要进行的点乘次数等于卷积核的参数数量
dot_product_count = kernel_channels * kernel_height * kernel_width

print(f"单个输出通道的一个像素值需要进行 {dot_product_count} 次点乘操作")

单个输出通道的一个像素值需要进行 75 次点乘操作


## 2.2 编程题

不使用深度学习框架的底层 Pooling API（如 `torch.nn.MaxPool2d`），仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向传播函数。

In [46]:
import numpy as np

def max_pool2d_forward(input_tensor, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    
    参数:
        input_tensor: 输入张量，形状为 (batch_size, channels, height, width)
        kernel_size: 池化核大小（int 或 tuple）
        stride: 步幅（int 或 tuple），默认为 1
        padding: 填充（int 或 tuple），默认为 0
    
    返回:
        output: 输出张量，形状为 (batch_size, channels, out_height, out_width)
    """
    # 处理 kernel_size 为 int 的情况
    if isinstance(kernel_size, int):
        kernel_h, kernel_w = kernel_size, kernel_size
    else:
        kernel_h, kernel_w = kernel_size
    
    # 处理 stride 为 int 的情况
    if isinstance(stride, int):
        stride_h, stride_w = stride, stride
    else:
        stride_h, stride_w = stride
    
    # 处理 padding 为 int 的情况
    if isinstance(padding, int):
        pad_h, pad_w = padding, padding
    else:
        pad_h, pad_w = padding
    
    # 获取输入形状
    batch_size, channels, in_height, in_width = input_tensor.shape
    
    # 计算输出尺寸
    out_height = (in_height + 2 * pad_h - kernel_h) // stride_h + 1
    out_width = (in_width + 2 * pad_w - kernel_w) // stride_w + 1
    
    # 初始化输出张量
    output = np.zeros((batch_size, channels, out_height, out_width))
    
    # 对输入进行 zero-padding
    padded_input = np.pad(input_tensor, 
                          ((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)),
                          mode='constant', constant_values=np.min(input_tensor))
    
    # 遍历每个样本
    for b in range(batch_size):
        # 遍历每个通道
        for c in range(channels):
            # 遍历输出高度
            for h_out in range(out_height):
                # 遍历输出宽度
                for w_out in range(out_width):
                    # 计算当前池化窗口在输入上的位置
                    h_start = h_out * stride_h
                    h_end = h_start + kernel_h
                    w_start = w_out * stride_w
                    w_end = w_start + kernel_w
                    
                    # 获取池化窗口内的区域
                    window = padded_input[b, c, h_start:h_end, w_start:w_end]
                    
                    # 取最大值
                    output[b, c, h_out, w_out] = np.max(window)
    
    return output

### 测试实现

In [47]:
# 创建测试输入
np.random.seed(42)
test_input = np.random.randint(0, 10, size=(2, 3, 8, 8))  # (batch, channels, height, width)

print("测试输入形状:", test_input.shape)
print("\n测试输入示例（第一个样本，第一个通道）:")
print(test_input[0, 0])

测试输入形状: (2, 3, 8, 8)

测试输入示例（第一个样本，第一个通道）:
[[6 3 7 4 6 9 2 6]
 [7 4 3 7 7 2 5 4]
 [1 7 5 1 4 0 9 5]
 [8 0 9 2 6 3 8 2]
 [4 2 6 4 8 6 1 3]
 [8 1 9 8 9 4 1 3]
 [6 7 2 0 3 1 7 3]
 [1 5 5 9 3 5 1 9]]


In [48]:
# 测试无 padding，默认 stride=1 的情况
output1 = max_pool2d_forward(test_input, kernel_size=2, stride=1, padding=0)
print("输出形状 (kernel=2, stride=1, padding=0):", output1.shape)
print("\n输出示例（第一个样本，第一个通道）:")
print(output1[0, 0])

输出形状 (kernel=2, stride=1, padding=0): (2, 3, 7, 7)

输出示例（第一个样本，第一个通道）:
[[7. 7. 7. 7. 9. 9. 6.]
 [7. 7. 7. 7. 7. 9. 9.]
 [8. 9. 9. 6. 6. 9. 9.]
 [8. 9. 9. 8. 8. 8. 8.]
 [8. 9. 9. 9. 9. 6. 3.]
 [8. 9. 9. 9. 9. 7. 7.]
 [7. 7. 9. 9. 5. 7. 9.]]


In [49]:
# 测试有 padding 和 stride 的情况
output2 = max_pool2d_forward(test_input, kernel_size=3, stride=2, padding=1)
print("输出形状 (kernel=3, stride=2, padding=1):", output2.shape)
print("\n输出示例（第一个样本，第一个通道）:")
print(output2[0, 0])

输出形状 (kernel=3, stride=2, padding=1): (2, 3, 4, 4)

输出示例（第一个样本，第一个通道）:
[[7. 7. 9. 9.]
 [8. 9. 7. 9.]
 [8. 9. 9. 8.]
 [8. 9. 9. 9.]]


### 与 PyTorch 官方实现对比验证

In [50]:
import torch
import torch.nn as nn

# 将 numpy 数组转换为 torch 张量
torch_input = torch.from_numpy(test_input).float()

# 使用 PyTorch 官方实现
maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
torch_output = maxpool(torch_input)

# 对比结果
print("PyTorch 输出形状:", torch_output.shape)
print("\nPyTorch 输出示例（第一个样本，第一个通道）:")
print(torch_output[0, 0])

# 验证是否一致
print("\n验证结果是否一致:", np.allclose(output2, torch_output.numpy()))

PyTorch 输出形状: torch.Size([2, 3, 4, 4])

PyTorch 输出示例（第一个样本，第一个通道）:
tensor([[7., 7., 9., 9.],
        [8., 9., 7., 9.],
        [8., 9., 9., 8.],
        [8., 9., 9., 9.]])

验证结果是否一致: True


## 3 LeNet, AlexNet, VGG 和 NiN

## 3.1 理论计算题

在 VGG 网络中，作者频繁使用多个 3×3 卷积核级联来代替较大的卷积核（如 5×5 或 7×7）。假设输入和输出的特征图通道数均为 C。

### 问题1：计算一个 5×5 卷积层（不带偏置）的参数量

In [51]:
# 定义通道数 C
C = 'C'  # 用符号表示

# 5x5 卷积层的参数量计算
# 输入通道数 = C, 输出通道数 = C, 卷积核大小 = 5x5
# 参数量 = 输出通道数 x 输入通道数 x 卷积核高度 x 卷积核宽度
print("5x5 卷积层参数量计算:")
print("= 输出通道数 x 输入通道数 x 卷积核高度 x 卷积核宽度")
print("= C x C x 5 x 5")
print("= 25C^2")

# 用具体数值示例验证
C_val = 64
params_5x5 = C_val * C_val * 5 * 5
print(f"\n当 C = {C_val} 时, 参数量 = {params_5x5}")

5x5 卷积层参数量计算:
= 输出通道数 x 输入通道数 x 卷积核高度 x 卷积核宽度
= C x C x 5 x 5
= 25C^2

当 C = 64 时, 参数量 = 102400


### 问题2：计算两个串联的 3×3 卷积层（不带偏置，两层通道数都为 C）的总参数量

In [52]:
# 第一个 3x3 卷积层：输入通道 C, 输出通道 C
# 第二个 3x3 卷积层：输入通道 C, 输出通道 C

print("两个串联的 3x3 卷积层总参数量计算:")
print("\n第一个 3x3 卷积层:")
print("  参数量 = C x C x 3 x 3 = 9C^2")

print("\n第二个 3x3 卷积层:")
print("  参数量 = C x C x 3 x 3 = 9C^2")

print("\n总参数量 = 9C^2 + 9C^2 = 18C^2")

# 用具体数值示例验证
C_val = 64
params_3x3_total = 2 * C_val * C_val * 3 * 3
print(f"\n当 C = {C_val} 时, 总参数量 = {params_3x3_total}")

# 对比
params_5x5 = C_val * C_val * 5 * 5
print(f"\n对比:")
print(f"5x5 卷积层参数量: {params_5x5}")
print(f"两个 3x3 卷积层参数量: {params_3x3_total}")
print(f"参数量减少: {params_5x5 - params_3x3_total} ({(params_5x5 - params_3x3_total) / params_5x5 * 100:.1f}%)")

两个串联的 3x3 卷积层总参数量计算:

第一个 3x3 卷积层:
  参数量 = C x C x 3 x 3 = 9C^2

第二个 3x3 卷积层:
  参数量 = C x C x 3 x 3 = 9C^2

总参数量 = 9C^2 + 9C^2 = 18C^2

当 C = 64 时, 总参数量 = 73728

对比:
5x5 卷积层参数量: 102400
两个 3x3 卷积层参数量: 73728
参数量减少: 28672 (28.0%)


## 3.2 编程题

NiN 网络的核心创新是引入了 "1x1 卷积" 组成的 NiN 块来代替传统的全连接层，以减少参数量。请使用 PyTorch（`torch.nn.Sequential`）定义一个标准的 NiN 块（NiN Block）。

**要求**：NiN 块接收输入通道数 `in_channels` 和输出通道数 `out_channels`，它由一个普通的卷积层（指定窗口大小 `kernel_size`，步幅 `stride`，填充 `padding`）以及两个随后的 1×1 卷积层级联组成。每层卷积后都需要紧跟一个 ReLU 激活层。

In [53]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义一个标准的 NiN (Network in Network) 块
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 第一个卷积层的卷积核大小
        stride: 第一个卷积层的步幅
        padding: 第一个卷积层的填充
    
    返回:
        nn.Sequential: 包含三层卷积+ReLU的 NiN 块
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

### 测试 NiN Block

In [54]:
# 创建一个 NiN 块实例
nin = nin_block(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)

# 打印网络结构
print("NiN 块结构:")
print(nin)

# 测试前向传播
test_input = torch.randn(1, 3, 224, 224)
output = nin(test_input)

print(f"\n输入形状: {test_input.shape}")
print(f"输出形状: {output.shape}")

NiN 块结构:
Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU()
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)

输入形状: torch.Size([1, 3, 224, 224])
输出形状: torch.Size([1, 96, 54, 54])


In [55]:
# 测试不同配置的 NiN 块
nin2 = nin_block(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2)

test_input2 = torch.randn(1, 96, 27, 27)
output2 = nin2(test_input2)

print("NiN 块 2 结构:")
print(nin2)
print(f"\n输入形状: {test_input2.shape}")
print(f"输出形状: {output2.shape}")

NiN 块 2 结构:
Sequential(
  (0): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)

输入形状: torch.Size([1, 96, 27, 27])
输出形状: torch.Size([1, 256, 27, 27])


## 4 Inception, 批量归一化和残差网络

## 4.1 理论计算题

在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：x1=2, x2=4, x3=6, x4=8。假设当前批量归一化层学到的缩放参数 gamma=2，平移参数 beta=1，常数 eps=0。

请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 y1, y2, y3, y4。

In [56]:
import numpy as np

# 输入数据
x = np.array([2.0, 4.0, 6.0, 8.0])

# Batch Normalization 参数
gamma = 2.0
beta = 1.0
epsilon = 0.0

print("Batch Normalization 计算过程:")
print("=" * 50)

# 步骤1: 计算均值
mu = np.mean(x)
print(f"步骤1: 计算均值")
print(f"  mu = (2 + 4 + 6 + 8) / 4 = {mu}")

# 步骤2: 计算方差
var = np.var(x, ddof=0)
print(f"\n步骤2: 计算方差")
print(f"  var = [(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2] / 4")
print(f"      = [9 + 1 + 1 + 9] / 4 = {var}")

# 步骤3: 标准化
x_hat = (x - mu) / np.sqrt(var + epsilon)
print(f"\n步骤3: 标准化")
for i in range(4):
    print(f"  x_hat_{i+1} = ({x[i]} - {mu}) / sqrt({var}) = {x_hat[i]:.4f}")

# 步骤4: 缩放和平移
y = gamma * x_hat + beta
print(f"\n步骤4: 缩放和平移")
for i in range(4):
    print(f"  y_{i+1} = {gamma} * {x_hat[i]:.4f} + {beta} = {y[i]:.4f}")

print("\n" + "=" * 50)
print("最终结果:")
for i in range(4):
    print(f"  y_{i+1} = {y[i]:.4f}")

Batch Normalization 计算过程:
步骤1: 计算均值
  mu = (2 + 4 + 6 + 8) / 4 = 5.0

步骤2: 计算方差
  var = [(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2] / 4
      = [9 + 1 + 1 + 9] / 4 = 5.0

步骤3: 标准化
  x_hat_1 = (2.0 - 5.0) / sqrt(5.0) = -1.3416
  x_hat_2 = (4.0 - 5.0) / sqrt(5.0) = -0.4472
  x_hat_3 = (6.0 - 5.0) / sqrt(5.0) = 0.4472
  x_hat_4 = (8.0 - 5.0) / sqrt(5.0) = 1.3416

步骤4: 缩放和平移
  y_1 = 2.0 * -1.3416 + 1.0 = -1.6833
  y_2 = 2.0 * -0.4472 + 1.0 = 0.1056
  y_3 = 2.0 * 0.4472 + 1.0 = 1.8944
  y_4 = 2.0 * 1.3416 + 1.0 = 3.6833

最终结果:
  y_1 = -1.6833
  y_2 = 0.1056
  y_3 = 1.8944
  y_4 = 3.6833


## 4.2 编程题

残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的梯度消失问题。请用 PyTorch 自定义一个残差块类 `Residual`。

**要求**：该块包含两个具有相同输出通道数的 3×3 卷积层，每个卷积层后跟一个批量归一化层。如果 `use_1x1conv=True`，则需要对输入应用一个 1×1 的卷积层来调整输入的通道数和形状，以便它能和第二层卷积的输出进行按元素相加。

In [57]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块 (Residual Block)
    
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        use_1x1conv: 是否使用 1x1 卷积调整输入维度
        stride: 第一个卷积层的步幅
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个 3x3 卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个 3x3 卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1x1 卷积调整层
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
        
        # ReLU 激活函数
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        identity = x
        
        # 第一个卷积 + BN + ReLU
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        # 第二个卷积 + BN
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 如果输入输出维度不匹配，使用 1x1 卷积调整
        if self.conv3 is not None:
            identity = self.conv3(x)
        
        # 残差连接
        out += identity
        out = self.relu(out)
        
        return out

### 测试 Residual 块

In [58]:
# 测试1: 输入输出通道数相同
print("测试1: 输入输出通道数相同")
res1 = Residual(in_channels=64, out_channels=64, use_1x1conv=False)
test_input1 = torch.randn(2, 64, 32, 32)
output1 = res1(test_input1)
print(f"输入形状: {test_input1.shape}")
print(f"输出形状: {output1.shape}")
print(f"形状是否一致: {test_input1.shape == output1.shape}")

测试1: 输入输出通道数相同
输入形状: torch.Size([2, 64, 32, 32])
输出形状: torch.Size([2, 64, 32, 32])
形状是否一致: True


In [59]:
# 测试2: 输入输出通道数不同
print("\n测试2: 输入输出通道数不同")
res2 = Residual(in_channels=64, out_channels=128, use_1x1conv=True, stride=2)
test_input2 = torch.randn(2, 64, 32, 32)
output2 = res2(test_input2)
print(f"输入形状: {test_input2.shape}")
print(f"输出形状: {output2.shape}")


测试2: 输入输出通道数不同
输入形状: torch.Size([2, 64, 32, 32])
输出形状: torch.Size([2, 128, 16, 16])


In [60]:
# 验证梯度流
print("\n验证梯度流:")
res_test = Residual(in_channels=3, out_channels=3, use_1x1conv=False)
x = torch.randn(1, 3, 4, 4, requires_grad=True)
y = res_test(x)
loss = y.sum()
loss.backward()
print(f"输入梯度是否存在: {x.grad is not None}")
print(f"输入梯度形状: {x.grad.shape}")
print("残差连接确保梯度可以直接回传到输入!")


验证梯度流:
输入梯度是否存在: True
输入梯度形状: torch.Size([1, 3, 4, 4])
残差连接确保梯度可以直接回传到输入!


## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

#### 问题1：为什么我们通常对除了最终输出层之外的底层特征提取层设置较小的学习率，而对新初始化的顶层输出层设置较大的学习率？

**解答**：

1. 底层特征的通用性：预训练模型的底层特征提取层学习到的是通用的低级特征，如边缘、纹理、颜色等，这些特征在不同任务中都具有很好的泛化能力。

2. 避免破坏预训练知识：如果对底层网络使用较大的学习率，可能会破坏已经学到的有用特征表示，导致模型性能下降。

3. 顶层需要快速适应：新初始化的顶层输出层需要快速学习目标任务的特定分类边界，因此需要较大的学习率来快速调整参数。

4. 迁移学习的核心思想：利用源数据集学到的通用特征，在目标数据集上进行微调，只需调整顶层以适应新任务即可。

#### 问题2：如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？

**解答**：

当目标数据集很小且与源数据集相似时，可以采取以下策略防止过拟合：

1. 冻结大部分底层网络：只训练顶层输出层，完全冻结预训练模型的特征提取部分。

2. 使用更小的学习率：即使微调部分层，也使用非常小的学习率，避免参数剧烈变化。

3. 数据增强：对目标数据集进行图像增广，如随机裁剪、翻转、颜色变化等，增加数据多样性。

4. 正则化：添加 Dropout 层或使用权重衰减来限制模型复杂度。

5. 早停：监控验证集性能，当性能不再提升时停止训练。

6. 使用预训练模型的中间特征：可以提取预训练模型的中间层特征作为输入，训练一个简单的分类器。

### 5.2 编程题

In [61]:
import torchvision.transforms as transforms

# 创建组合图像增广管道
train_transform = transforms.Compose([
    # 1. 随机裁剪，面积比例在 0.08 到 1.0 之间，然后缩放到 224x224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度和饱和度，变化范围为 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor()
])

print("图像增广管道创建成功!")
print(train_transform)

图像增广管道创建成功!
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


In [62]:
from PIL import Image

# 创建一个测试图像
test_image = Image.new('RGB', (256, 256), color=(128, 128, 128))

# 应用增广管道
transformed_tensor = train_transform(test_image)

print(f"原始图像形状: ({test_image.mode}, {test_image.size[0]}x{test_image.size[1]})")
print(f"转换后张量形状: {transformed_tensor.shape}")
print(f"张量数据类型: {transformed_tensor.dtype}")

原始图像形状: (RGB, 256x256)
转换后张量形状: torch.Size([3, 224, 224])
张量数据类型: torch.float32


## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

**题目**：计算两个边界框之间的 IoU（交并比）。

- 真实框 A = [10, 10, 50, 50]（左上角 x, 左上角 y, 右下角 x, 右下角 y）
- 预测框 B = [30, 30, 70, 70]

**IoU 计算过程**：

1. **计算交集区域**：
   - 交集左上角: max(A_x1, B_x1) = max(10, 30) = 30
   - 交集左上角: max(A_y1, B_y1) = max(10, 30) = 30
   - 交集右下角: min(A_x2, B_x2) = min(50, 70) = 50
   - 交集右下角: min(A_y2, B_y2) = min(50, 70) = 50
   - 交集宽度: 50 - 30 = 20
   - 交集高度: 50 - 30 = 20
   - 交集面积: 20 * 20 = 400

2. **计算并集区域**：
   - A 的面积: (50-10) * (50-10) = 40 * 40 = 1600
   - B 的面积: (70-30) * (70-30) = 40 * 40 = 1600
   - 并集面积: A + B - 交集 = 1600 + 1600 - 400 = 2800

3. **计算 IoU**：
   - IoU = 交集面积 / 并集面积 = 400 / 2800 = 1/7 ≈ 0.1429

In [63]:
# IoU 计算代码
def calculate_iou(box1, box2):
    """
    计算两个边界框的 IoU
    box: [x1, y1, x2, y2]
    """
    x1, y1, x2, y2 = box1
    x1_p, y1_p, x2_p, y2_p = box2
    
    # 计算交集
    inter_x1 = max(x1, x1_p)
    inter_y1 = max(y1, y1_p)
    inter_x2 = min(x2, x2_p)
    inter_y2 = min(y2, y2_p)
    
    # 计算交集面积
    inter_width = max(0, inter_x2 - inter_x1)
    inter_height = max(0, inter_y2 - inter_y1)
    inter_area = inter_width * inter_height
    
    # 计算各自面积
    area1 = (x2 - x1) * (y2 - y1)
    area2 = (x2_p - x1_p) * (y2_p - y1_p)
    
    # 计算并集面积
    union_area = area1 + area2 - inter_area
    
    # 计算 IoU
    iou = inter_area / union_area if union_area > 0 else 0.0
    
    return iou

# 测试
A = [10, 10, 50, 50]
B = [30, 30, 70, 70]

iou = calculate_iou(A, B)
print(f"边界框 A: {A}")
print(f"边界框 B: {B}")
print(f"IoU 值: {iou:.4f}")

边界框 A: [10, 10, 50, 50]
边界框 B: [30, 30, 70, 70]
IoU 值: 0.1429


### 6.2 编程题

**题目**：实现一个计算标签平滑后交叉熵损失的函数。

标签平滑（Label Smoothing）通过防止模型过于自信地预测某些类别来提高泛化性。

标准交叉熵使用独热编码，若设置平滑因子 ε = 0.1，则对于 K 分类问题：
- 真实标签对应的目标概率从 1 变为 1 - ε
- 其余错误类别的概率从 0 变为 ε / (K - 1)

In [64]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    计算标签平滑后的交叉熵损失
    
    参数:
        logits: 模型输出的 logits，形状为 [batch_size, num_classes]
        targets: 真实标签，形状为 [batch_size]
        epsilon: 平滑因子，默认 0.1
    
    返回:
        平均交叉熵损失
    """
    num_classes = logits.size(-1)
    
    # 创建平滑后的目标分布
    smooth_targets = torch.full_like(logits, epsilon / (num_classes - 1))
    smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - epsilon)
    
    # 计算对数概率
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 计算交叉熵损失
    loss = -(smooth_targets * log_probs).sum(dim=-1)
    
    return loss.mean()

In [65]:
# 测试标签平滑交叉熵损失

# 模拟输入
batch_size = 3
num_classes = 5

# 随机生成 logits 和标签
logits = torch.randn(batch_size, num_classes)
targets = torch.tensor([0, 2, 4])  # 真实标签

# 计算标签平滑后的交叉熵损失
loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)

print(f"Logits 形状: {logits.shape}")
print(f"Targets: {targets}")
print(f"标签平滑交叉熵损失: {loss.item():.4f}")

# 与标准交叉熵对比
standard_loss = F.cross_entropy(logits, targets)
print(f"标准交叉熵损失: {standard_loss.item():.4f}")

Logits 形状: torch.Size([3, 5])
Targets: tensor([0, 2, 4])
标签平滑交叉熵损失: 2.0551
标准交叉熵损失: 2.0635
